In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
#df = spark.read.format("csv").option("header", "True").load("file:///home/talentum/Big_Data_Project_Work/Big_Data_Project/flights_sample_2m.csv")

In [4]:
df = spark.read.parquet(
    "hdfs:///user/hadoop/aviation/landing_zone")

In [5]:
df.count()

2000000

In [6]:
rows = df.count()
cols = len(df.columns)

print(f"Dimensions: ({rows}, {cols})")

Dimensions: (2000000, 32)


In [7]:
df.columns

['FL_DATE',
 'AIRLINE',
 'AIRLINE_DOT',
 'AIRLINE_CODE',
 'DOT_CODE',
 'FL_NUMBER',
 'ORIGIN',
 'ORIGIN_CITY',
 'DEST',
 'DEST_CITY',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'WHEELS_ON',
 'TAXI_IN',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'CRS_ELAPSED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'DELAY_DUE_CARRIER',
 'DELAY_DUE_WEATHER',
 'DELAY_DUE_NAS',
 'DELAY_DUE_SECURITY',
 'DELAY_DUE_LATE_AIRCRAFT']

In [8]:
df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- AIRLINE_DOT: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- DOT_CODE: double (nullable = true)
 |-- FL_NUMBER: double (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 

### Adding the flight ID column

In [9]:
from pyspark.sql.functions import monotonically_increasing_id

In [10]:
df = df.withColumn("flight_id", \
                  monotonically_increasing_id())

In [11]:
df.select("flight_id").show(20)

+---------+
|flight_id|
+---------+
|        0|
|        1|
|        2|
|        3|
|        4|
|        5|
|        6|
|        7|
|        8|
|        9|
|       10|
|       11|
|       12|
|       13|
|       14|
|       15|
|       16|
|       17|
|       18|
|       19|
+---------+
only showing top 20 rows



### Creating Airline Dimension Table

In [12]:
airline_dim = df.select(
            "AIRLINE",\
            "AIRLINE_DOT",\
            "AIRLINE_CODE",\
            "DOT_CODE").distinct()

In [13]:
airline_dim.show(10)

+--------------------+--------------------+------------+--------+
|             AIRLINE|         AIRLINE_DOT|AIRLINE_CODE|DOT_CODE|
+--------------------+--------------------+------------+--------+
|           Envoy Air|       Envoy Air: MQ|          MQ| 20398.0|
|Delta Air Lines Inc.|Delta Air Lines I...|          DL| 19790.0|
|     JetBlue Airways| JetBlue Airways: B6|          B6| 20409.0|
|  Mesa Airlines Inc.|Mesa Airlines Inc...|          YV| 20378.0|
|Hawaiian Airlines...|Hawaiian Airlines...|          HA| 19690.0|
|Frontier Airlines...|Frontier Airlines...|          F9| 20436.0|
|   PSA Airlines Inc.|PSA Airlines Inc....|          OH| 20397.0|
|American Airlines...|American Airlines...|          AA| 19805.0|
|         Horizon Air|     Horizon Air: QX|          QX| 19687.0|
|United Air Lines ...|United Air Lines ...|          UA| 19977.0|
+--------------------+--------------------+------------+--------+
only showing top 10 rows



In [14]:
airline_dim.count()

18

### Splitting the origin flights

In [15]:
origin_flights = df.select(
                    "ORIGIN",\
                    "ORIGIN_CITY").distinct()

In [16]:
origin_flights.show(10)

+------+--------------------+
|ORIGIN|         ORIGIN_CITY|
+------+--------------------+
|   COS|Colorado Springs, CO|
|   SDF|      Louisville, KY|
|   CLL|College Station/B...|
|   PIR|          Pierre, SD|
|   MSN|         Madison, WI|
|   CMI|Champaign/Urbana, IL|
|   EVV|      Evansville, IN|
|   GTR|        Columbus, MS|
|   RIC|        Richmond, VA|
|   STX|   Christiansted, VI|
+------+--------------------+
only showing top 10 rows



In [17]:
origin_flights.count()

381

### Splitting the destination flights

In [18]:
dest_flights = df.select(
                        "DEST",\
                        "DEST_CITY"\
                        ).distinct()

In [19]:
dest_flights.show(10)

+----+--------------------+
|DEST|           DEST_CITY|
+----+--------------------+
| COS|Colorado Springs, CO|
| SDF|      Louisville, KY|
| PIR|          Pierre, SD|
| CLL|College Station/B...|
| MSN|         Madison, WI|
| EVV|      Evansville, IN|
| GTR|        Columbus, MS|
| CMI|Champaign/Urbana, IL|
| RIC|        Richmond, VA|
| STX|   Christiansted, VI|
+----+--------------------+
only showing top 10 rows



### Renaming the columns in Origin and Dest flights tables

In [20]:
from pyspark.sql.functions import col

In [21]:
from pyspark.sql.functions import upper, trim

In [22]:
origin_airports = origin_flights.select(
                                    col("ORIGIN").alias("AIRPORT_CODE"),\
                                    upper(trim(col("ORIGIN_CITY"))).alias("CITY")
                                       )

In [23]:
origin_airports.show(10)

+------------+--------------------+
|AIRPORT_CODE|                CITY|
+------------+--------------------+
|         COS|COLORADO SPRINGS, CO|
|         SDF|      LOUISVILLE, KY|
|         CLL|COLLEGE STATION/B...|
|         PIR|          PIERRE, SD|
|         MSN|         MADISON, WI|
|         CMI|CHAMPAIGN/URBANA, IL|
|         EVV|      EVANSVILLE, IN|
|         GTR|        COLUMBUS, MS|
|         RIC|        RICHMOND, VA|
|         STX|   CHRISTIANSTED, VI|
+------------+--------------------+
only showing top 10 rows



In [24]:
dest_airports = dest_flights.select(
                                    col("DEST").alias("AIRPORT_CODE"),\
                                    upper(trim(col("DEST_CITY"))).alias("CITY")
                                       )

In [25]:
dest_airports.show(10)

+------------+--------------------+
|AIRPORT_CODE|                CITY|
+------------+--------------------+
|         COS|COLORADO SPRINGS, CO|
|         SDF|      LOUISVILLE, KY|
|         PIR|          PIERRE, SD|
|         CLL|COLLEGE STATION/B...|
|         MSN|         MADISON, WI|
|         EVV|      EVANSVILLE, IN|
|         GTR|        COLUMBUS, MS|
|         CMI|CHAMPAIGN/URBANA, IL|
|         RIC|        RICHMOND, VA|
|         STX|   CHRISTIANSTED, VI|
+------------+--------------------+
only showing top 10 rows



### Combining both tables into single table

In [26]:
airport_dim = (
             origin_airports.\
             union(dest_airports).\
            distinct())

In [27]:
airport_dim.show(10)

+------------+-----------------+
|AIRPORT_CODE|             CITY|
+------------+-----------------+
|         PGD|  PUNTA GORDA, FL|
|         DAB|DAYTONA BEACH, FL|
|         CWA|      MOSINEE, WI|
|         ESC|     ESCANABA, MI|
|         LAR|      LARAMIE, WY|
|         BZN|      BOZEMAN, MT|
|         TUS|       TUCSON, AZ|
|         JNU|       JUNEAU, AK|
|         CHA|  CHATTANOOGA, TN|
|         DCA|   WASHINGTON, DC|
+------------+-----------------+
only showing top 10 rows



In [28]:
airport_dim.count()

380

In [29]:
airport_dim.select(
    "AIRPORT_CODE"
).distinct().count()

380

In [30]:
delay_fact = df.select(
            "flight_id",\
            "DELAY_DUE_CARRIER",\
            "DELAY_DUE_WEATHER",\
            "DELAY_DUE_NAS",\
            "DELAY_DUE_SECURITY",\
            "DELAY_DUE_LATE_AIRCRAFT")

In [31]:
delay_fact.show(10)

+---------+-----------------+-----------------+-------------+------------------+-----------------------+
|flight_id|DELAY_DUE_CARRIER|DELAY_DUE_WEATHER|DELAY_DUE_NAS|DELAY_DUE_SECURITY|DELAY_DUE_LATE_AIRCRAFT|
+---------+-----------------+-----------------+-------------+------------------+-----------------------+
|        0|             null|             null|         null|              null|                   null|
|        1|             null|             null|         null|              null|                   null|
|        2|             null|             null|         null|              null|                   null|
|        3|             53.0|              0.0|          0.0|               0.0|                    0.0|
|        4|             63.0|              0.0|         97.0|               0.0|                    0.0|
|        5|             null|             null|         null|              null|                   null|
|        6|             null|             null|        

In [32]:
delay_fact.printSchema()

root
 |-- flight_id: long (nullable = false)
 |-- DELAY_DUE_CARRIER: double (nullable = true)
 |-- DELAY_DUE_WEATHER: double (nullable = true)
 |-- DELAY_DUE_NAS: double (nullable = true)
 |-- DELAY_DUE_SECURITY: double (nullable = true)
 |-- DELAY_DUE_LATE_AIRCRAFT: double (nullable = true)



In [33]:
delay_fact.count()

2000000

In [34]:

delay_fact.filter(
    col("DELAY_DUE_CARRIER").isNotNull()
).count()

356220

### Calculating non-null values in these columns

In [35]:
total = delay_fact.count()

In [36]:
carrier = delay_fact.filter(
            col("DELAY_DUE_CARRIER").isNotNull()).count()

In [37]:
weather = delay_fact.filter(
            col("DELAY_DUE_WEATHER").isNotNull()).count()

In [38]:
nas = delay_fact.filter(
    col("DELAY_DUE_NAS").isNotNull()
).count()

In [39]:
security = delay_fact.filter(
    col("DELAY_DUE_SECURITY").isNotNull()
).count()

In [40]:
late_aircraft = delay_fact.filter(
    col("DELAY_DUE_LATE_AIRCRAFT").isNotNull()
).count()

In [41]:
print("Carrier:", carrier)
print("Weather:", weather)
print("NAS:", nas)
print("Security:", security)
print("Late Aircraft:", late_aircraft)

Carrier: 356220
Weather: 356220
NAS: 356220
Security: 356220
Late Aircraft: 356220


### Creating flight fact table

In [42]:
flight_fact = df.select(
    "flight_id",
    "FL_DATE",
    "AIRLINE_CODE",
    "FL_NUMBER",
    col("ORIGIN").alias("ORIGIN_AIRPORT_CODE"),
    col("DEST").alias("DEST_AIRPORT_CODE"),
    "CRS_DEP_TIME",
    "DEP_TIME",
    "CRS_ARR_TIME",
    "ARR_TIME",
    "DEP_DELAY",
    "ARR_DELAY",
    "DISTANCE",
    "CANCELLED",
    "DIVERTED"
)

In [43]:
flight_fact.show(10)

+---------+----------+------------+---------+-------------------+-----------------+------------+--------+------------+--------+---------+---------+--------+---------+--------+
|flight_id|   FL_DATE|AIRLINE_CODE|FL_NUMBER|ORIGIN_AIRPORT_CODE|DEST_AIRPORT_CODE|CRS_DEP_TIME|DEP_TIME|CRS_ARR_TIME|ARR_TIME|DEP_DELAY|ARR_DELAY|DISTANCE|CANCELLED|DIVERTED|
+---------+----------+------------+---------+-------------------+-----------------+------------+--------+------------+--------+---------+---------+--------+---------+--------+
|        0|2021-06-24|          AA|   2708.0|                DFW|              LGA|      1844.0|  1841.0|      2325.0|  2300.0|     -3.0|    -25.0|  1389.0|      0.0|     0.0|
|        1|2019-02-24|          OO|   4321.0|                ABE|              DTW|       545.0|   542.0|       750.0|   728.0|     -3.0|    -22.0|   425.0|      0.0|     0.0|
|        2|2023-08-06|          MQ|   3777.0|                AUS|              SMF|       859.0|   909.0|      1034.0|  

In [44]:
flight_fact.printSchema()

root
 |-- flight_id: long (nullable = false)
 |-- FL_DATE: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- FL_NUMBER: double (nullable = true)
 |-- ORIGIN_AIRPORT_CODE: string (nullable = true)
 |-- DEST_AIRPORT_CODE: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)



In [45]:
flight_fact.count()

2000000

### Checking Airline Join

In [46]:
print(flight_fact.select("AIRLINE_CODE").distinct().count())

print(airline_dim.select("AIRLINE_CODE").distinct().count())

18
18


### Checking Airport Join

In [47]:
print(flight_fact.select("ORIGIN_AIRPORT_CODE").distinct().count())

print(airport_dim.select("AIRPORT_CODE").distinct().count())

380
380


### Checking Delay Join

In [48]:
print(flight_fact.count())

print(delay_fact.count())

2000000
2000000


In [49]:
flight_fact.select("flight_id").distinct().count()

2000000

In [50]:
delay_fact.select("flight_id").distinct().count()

2000000

### Saving the dataframes

In [69]:
flight_fact.coalesce(10).write.mode("overwrite").parquet("hdfs:///user/hadoop/aviation/final_parquets/flight_fact")

In [70]:
airline_dim.coalesce(1) \
.write \
.mode("overwrite") \
.parquet("hdfs:///user/hadoop/aviation/final_parquets/airline_dim")

In [68]:
delay_fact.coalesce(10).write.mode("overwrite").parquet("hdfs:///user/hadoop/aviation/final_parquets/delay_fact")

In [71]:
airport_dim.coalesce(1) \
.write \
.mode("overwrite") \
.parquet("hdfs:///user/hadoop/aviation/final_parquets/airport_dim")

In [67]:
airline_dim.show(5)

+--------------------+--------------------+------------+--------+
|             AIRLINE|         AIRLINE_DOT|AIRLINE_CODE|DOT_CODE|
+--------------------+--------------------+------------+--------+
|           Envoy Air|       Envoy Air: MQ|          MQ| 20398.0|
|Delta Air Lines Inc.|Delta Air Lines I...|          DL| 19790.0|
|     JetBlue Airways| JetBlue Airways: B6|          B6| 20409.0|
|  Mesa Airlines Inc.|Mesa Airlines Inc...|          YV| 20378.0|
|Hawaiian Airlines...|Hawaiian Airlines...|          HA| 19690.0|
+--------------------+--------------------+------------+--------+
only showing top 5 rows



In [60]:
# ==========================================================
# Phase 1 Summary Report
# ==========================================================
# Generates Logs/phase1_summary.txt after all four parquet
# datasets have been written successfully.
# No transformations are changed — this is append-only.
# ==========================================================

import os
import time
from datetime import datetime

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

LANDING_ZONE      = "hdfs:///user/hadoop/aviation/landing_zone"
OUTPUT_HDFS       = "hdfs:///user/hadoop/aviation/"  # parent of all four outputs
LOG_DIR           = "/home/talentum/Big_Data_Project_Work/Big_Data_Project/Logs"
LOG_FILE          = os.path.join(LOG_DIR, "phase1_summary.txt")

# ----------------------------------------------------------
# Ensure Log Directory Exists
# ----------------------------------------------------------

os.makedirs(LOG_DIR, exist_ok=True)

# ----------------------------------------------------------
# Capture Execution Start Time
# ----------------------------------------------------------

phase1_start = time.time()
exec_dt      = datetime.now()

print("\n")
print("=" * 60)
print("         PHASE 1 — GENERATING SUMMARY REPORT")
print("=" * 60)
print("Counting records in each dataset...")
print("=" * 60)

# ----------------------------------------------------------
# Count Records In Each Generated Dataset
# Uses the DataFrames already in scope from earlier cells.
# ----------------------------------------------------------

flight_fact_count   = flight_fact.count()
airline_dim_count   = airline_dim.count()
airport_dim_count   = airport_dim.count()
delay_fact_count    = delay_fact.count()

phase1_elapsed = time.time() - phase1_start

# ----------------------------------------------------------
# Terminal Confirmation
# ----------------------------------------------------------

print("Flight Fact Count     :", flight_fact_count)
print("Airline Dim Count     :", airline_dim_count)
print("Airport Dim Count     :", airport_dim_count)
print("Delay Fact Count      :", delay_fact_count)
print("=" * 60)

# ----------------------------------------------------------
# Write Phase 1 Summary Log
# ----------------------------------------------------------

with open(LOG_FILE, "w") as log:
    log.write("=" * 60 + "\n")
    log.write("         AVIATION PIPELINE — PHASE 1 SUMMARY\n")
    log.write("=" * 60 + "\n")
    log.write("Title                  : Phase 1 — Data Splitting & Analysis\n")
    log.write("Execution Date         : {}\n".format(exec_dt.strftime("%d-%m-%Y")))
    log.write("Execution Time         : {}\n".format(exec_dt.strftime("%H:%M:%S")))
    log.write("Landing Zone Path      : {}\n".format(LANDING_ZONE))
    log.write("Output HDFS Location   : {}\n".format(OUTPUT_HDFS))
    log.write("Flight Fact Count      : {}\n".format(flight_fact_count))
    log.write("Airline Dim Count      : {}\n".format(airline_dim_count))
    log.write("Airport Dim Count      : {}\n".format(airport_dim_count))
    log.write("Delay Fact Count       : {}\n".format(delay_fact_count))
    log.write("Overall Status         : SUCCESS\n")
    log.write("=" * 60 + "\n")

print("Log saved to          :", LOG_FILE)
print("Status                : SUCCESS")
print("=" * 60)




         PHASE 1 — GENERATING SUMMARY REPORT
Counting records in each dataset...
Flight Fact Count     : 2000000
Airline Dim Count     : 18
Airport Dim Count     : 380
Delay Fact Count      : 2000000
Log saved to          : /home/talentum/Big_Data_Project_Work/Big_Data_Project/Logs/phase1_summary.txt
Status                : SUCCESS


In [61]:
phase1_elapsed

154.38557314872742